# Chapter 5 — Symmetric Operators and the SVD

Companion notebook to Chapter 5 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Conventions of the book: vectors are columns, `M` is the Gram matrix of the
inner product of the source space, `N` that of the target space, and the
adjoint of `A` has the matrix `M^{-1} A^T N`.

In [1]:
import math
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)

2.15.0+cu132


## 1. Signature: what survives `G -> P^T G P`

Sylvester's law of inertia (Theorem 5.3): under a change of basis the array
of a quadratic form becomes `P^T G P`. Its eigenvalues change; the numbers
of positive, negative and zero ones do not.

In [2]:
def signature(G, tol=1e-9):
    lam = torch.linalg.eigvalsh(G)
    return int((lam > tol).sum()), int((lam < -tol).sum())

# Example 5.5: x1*x2 and x1^2 + 4 x1 x2 + x2^2
G1 = torch.tensor([[0.0, 0.5], [0.5, 0.0]])
G2 = torch.tensor([[1.0, 2.0], [2.0, 1.0]])
print("x1 x2                 :", signature(G1))
print("x1^2 + 4 x1 x2 + x2^2 :", signature(G2))
assert signature(G1) == (1, 1) and signature(G2) == (1, 1)

# Exercise 5.1: x1 x2 + x2 x3 + x1 x3
G3 = 0.5 * (torch.ones(3, 3) - torch.eye(3))
print("x1x2 + x2x3 + x1x3    :", signature(G3),
      " eigenvalues", torch.linalg.eigvalsh(G3).tolist())
assert signature(G3) == (1, 2)

# Random changes of basis: eigenvalues move, the signature does not
G = torch.diag(torch.tensor([3.0, 1.0, -2.0, 0.0]))
for trial in range(5):
    P = torch.randn(4, 4)
    Gp = P.T @ G @ P
    assert signature(Gp, tol=1e-8) == (2, 1)
print("after P^T G P, eigenvalues:",
      [round(v, 3) for v in torch.linalg.eigvalsh(Gp).tolist()],
      "-> signature", signature(Gp, tol=1e-8))

x1 x2                 : (1, 1)
x1^2 + 4 x1 x2 + x2^2 : (1, 1)
x1x2 + x2x3 + x1x3    : (1, 2)  eigenvalues [-0.5000000000000002, -0.49999999999999983, 0.9999999999999998]
after P^T G P, eigenvalues: [-9.229, -0.0, 2.143, 9.301] -> signature (2, 1)


## 2. The same form, two inner products, two sets of eigenvalues

Proposition 5.6: with an inner product `M`, the form with array `G` becomes
the operator `S = M^{-1} G`, self-adjoint for `M`. Exercise 5.2 uses
`H = [[2, 1], [1, 2]]` with the dot product and with `M = diag(1, 4)`.

In [3]:
H = torch.tensor([[2.0, 1.0], [1.0, 2.0]])
M = torch.diag(torch.tensor([1.0, 4.0]))

def metric_eig(G, M):
    # eigenvalues/eigenvectors of M^{-1} G, eigenvectors M-orthonormal
    Lc = torch.linalg.cholesky(M)                  # M = Lc Lc^T
    Li = torch.linalg.inv(Lc)
    lam, Y = torch.linalg.eigh(Li @ G @ Li.T)
    P = Li.T @ Y
    return lam.flip(0), P.flip(1)

lam_I, _ = metric_eig(H, torch.eye(2))
lam_M, P = metric_eig(H, M)
print("dot product : eigenvalues", lam_I.tolist(),
      " kappa =", (lam_I[0] / lam_I[1]).item())
print("M = diag(1,4): eigenvalues", [round(v, 4) for v in lam_M.tolist()],
      " kappa =", round((lam_M[0] / lam_M[1]).item(), 3))
assert torch.allclose(lam_I, torch.tensor([3.0, 1.0]))
assert torch.allclose(lam_M, torch.tensor([(2.5 + math.sqrt(3.25)) / 2,
                                           (2.5 - math.sqrt(3.25)) / 2]))

# S = M^{-1} H is self-adjoint for M:  M S is symmetric, S itself is not
S = torch.linalg.solve(M, H)
print("S =", S.tolist())
assert torch.allclose(M @ S, (M @ S).T) and not torch.allclose(S, S.T)

# Corollary 5.8(2): simultaneous diagonalization
print("P^T M P =", (P.T @ M @ P).round(decimals=12).tolist())
print("P^T H P =", (P.T @ H @ P).round(decimals=4).tolist())
assert torch.allclose(P.T @ M @ P, torch.eye(2), atol=1e-12)
assert torch.allclose(P.T @ H @ P, torch.diag(lam_M), atol=1e-12)

dot product : eigenvalues [3.0, 1.0]  kappa = 3.0
M = diag(1,4): eigenvalues [2.1514, 0.3486]  kappa = 6.171
S = [[2.0, 1.0], [0.25, 0.5]]
P^T M P = [[1.0, 0.0], [0.0, 1.0]]
P^T H P = [[2.1514, 0.0], [0.0, 0.3486]]


## 3. Courant–Fischer

Theorem 5.9: on the span of the first `k` eigenvectors the minimum of the
Rayleigh quotient is `lambda_k`, and no subspace of dimension `k` does
better.

In [4]:
n, k = 6, 3
B = torch.randn(n, n)
S = (B + B.T) / 2
lam, E = torch.linalg.eigh(S)
lam, E = lam.flip(0), E.flip(1)

def min_rayleigh(S, U):
    # minimum of <v, S v> over unit vectors of the column span of U
    Q, _ = torch.linalg.qr(U)
    return torch.linalg.eigvalsh(Q.T @ S @ Q)[0].item()

best = min_rayleigh(S, E[:, :k])
print("lambda_k                    :", round(lam[k - 1].item(), 6))
print("min on span(e_1..e_k)       :", round(best, 6))
others = [min_rayleigh(S, torch.randn(n, k)) for _ in range(2000)]
print("largest min over 2000 random subspaces:", round(max(others), 6))
assert abs(best - lam[k - 1].item()) < 1e-10
assert max(others) <= lam[k - 1].item() + 1e-10

lambda_k                    : 1.23673
min on span(e_1..e_k)       : 1.23673
largest min over 2000 random subspaces: 1.040476


## 4. Gradient descent on a quadratic

Theorem 5.11 and Example 5.12, with `L(x) = x1^2 + 4 x2^2`, `H = diag(2, 8)`
and the start `x0 = (2.5, 1)` of Figure 5.2. The gradient is computed by
autograd; the metric update is `x - eta * M^{-1} grad`.

In [5]:
Hq = torch.diag(torch.tensor([2.0, 8.0]))

def loss(x):
    return x[0] ** 2 + 4 * x[1] ** 2

def run(x0, eta, steps, M=None):
    x = x0.clone()
    path = [x.clone()]
    for _ in range(steps):
        xr = x.clone().requires_grad_(True)
        (g,) = torch.autograd.grad(loss(xr), xr)
        step = g if M is None else torch.linalg.solve(M, g)
        x = x - eta * step
        path.append(x.clone())
    return torch.stack(path)

x0 = torch.tensor([2.5, 1.0])
path = run(x0, 0.2, 40)
print("first iterates, eta = 0.2:")
for p in path[:4]:
    print("  ", [round(v, 4) for v in p.tolist()])
assert torch.allclose(path[1], torch.tensor([1.5, -0.6]))
assert torch.allclose(path[2], torch.tensor([0.9, 0.36]))
assert torch.allclose(path[3], torch.tensor([0.54, -0.216]))

norms = path.norm(dim=1)
ratios = norms[1:] / norms[:-1]
print("error ratio per step:", round(ratios[0].item(), 6),
      "(theory: (kappa-1)/(kappa+1) = 0.6)")
assert torch.allclose(ratios, torch.full_like(ratios, 0.6))

# Exercise 5.4: 28 steps for a factor 1e-6
k = math.ceil(6 * math.log(10) / math.log(1 / 0.6))
print("steps for 1e-6:", k, " 0.6^27 =", f"{0.6**27:.3e}",
      " 0.6^28 =", f"{0.6**28:.3e}")
assert k == 28 and 0.6 ** 28 <= 1e-6 < 0.6 ** 27

# threshold eta = 2 / lambda_1 = 0.25
p25 = run(x0, 0.25, 6)
p30 = run(x0, 0.30, 6)
print("eta = 0.25, x2:", [round(v, 3) for v in p25[:, 1].tolist()])
print("eta = 0.30, x2:", [round(v, 3) for v in p30[:, 1].tolist()])
assert torch.allclose(p25[:, 1].abs(), torch.ones(7))
assert torch.allclose(p30[-1, 1].abs(), torch.tensor(1.4 ** 6))

# the inner product M = H: one step
p_newton = run(x0, 1.0, 1, M=Hq)
print("M = H, eta = 1, after one step:", (p_newton[1] + 0.0).tolist())
assert torch.allclose(p_newton[1], torch.zeros(2), atol=1e-14)

first iterates, eta = 0.2:
   [2.5, 1.0]
   [1.5, -0.6]
   [0.9, 0.36]
   [0.54, -0.216]
error ratio per step: 0.6 (theory: (kappa-1)/(kappa+1) = 0.6)
steps for 1e-6: 28  0.6^27 = 1.023e-06  0.6^28 = 6.141e-07
eta = 0.25, x2: [1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
eta = 0.30, x2: [1.0, -1.4, 1.96, -2.744, 3.842, -5.378, 7.53]
M = H, eta = 1, after one step: [0.0, 0.0]


The best rate over all step sizes: scan `eta` and compare with
`2 / (lambda_1 + lambda_n)`.

In [6]:
lam1, lamn = 8.0, 2.0
etas = torch.linspace(0.01, 0.3, 2901)
rates = torch.maximum((1 - etas * lam1).abs(), (1 - etas * lamn).abs())
i = rates.argmin()
print("best eta on the grid:", round(etas[i].item(), 4),
      " rate:", round(rates[i].item(), 4))
assert abs(etas[i].item() - 0.2) < 1e-3 and abs(rates[i].item() - 0.6) < 1e-3

best eta on the grid: 0.2  rate: 0.6


## 5. The singular value decomposition

Figure 5.3 and Exercise 5.5: `A = [[3, 0], [4, 5]]` for the dot products.

In [7]:
A = torch.tensor([[3.0, 0.0], [4.0, 5.0]])
U, s, Vh = torch.linalg.svd(A)
print("singular values:", s.tolist(), " (3*sqrt(5), sqrt(5)) =",
      (3 * math.sqrt(5), math.sqrt(5)))
assert torch.allclose(s, torch.tensor([3 * math.sqrt(5), math.sqrt(5)]))

# singular vectors are defined up to a common sign: compare projectors
v1 = torch.tensor([1.0, 1.0]) / math.sqrt(2)
w1 = torch.tensor([1.0, 3.0]) / math.sqrt(10)
v2 = torch.tensor([1.0, -1.0]) / math.sqrt(2)
w2 = torch.tensor([3.0, -1.0]) / math.sqrt(10)
assert torch.allclose(A @ v1, s[0] * w1) and torch.allclose(A @ v2, s[1] * w2)
assert torch.allclose(A.T @ w1, s[0] * v1) and torch.allclose(A.T @ w2, s[1] * v2)
assert torch.allclose(torch.outer(Vh[0], Vh[0]), torch.outer(v1, v1))
assert torch.allclose(torch.outer(U[:, 0], U[:, 0]), torch.outer(w1, w1))

print("spectral norm     :", torch.linalg.matrix_norm(A, ord=2).item())
print("condition number  :", torch.linalg.cond(A).item())
assert abs(torch.linalg.cond(A).item() - 3) < 1e-12

A1 = s[0] * torch.outer(w1, v1)
print("best rank-1 approximation:", A1.tolist())
print("errors (operator, HS):",
      torch.linalg.matrix_norm(A - A1, ord=2).item(),
      torch.linalg.matrix_norm(A - A1).item())
assert torch.allclose(A1, torch.tensor([[1.5, 1.5], [4.5, 4.5]]))
assert abs(torch.linalg.matrix_norm(A - A1, ord=2).item() - math.sqrt(5)) < 1e-12
assert abs(torch.linalg.matrix_norm(A - A1).item() - math.sqrt(5)) < 1e-12

singular values: [6.70820393249937, 2.2360679774997894]  (3*sqrt(5), sqrt(5)) = (6.708203932499369, 2.23606797749979)
spectral norm     : 6.70820393249937
condition number  : 3.000000000000001
best rank-1 approximation: [[1.5000000000000002, 1.5000000000000002], [4.5, 4.5]]
errors (operator, HS): 2.23606797749979 2.23606797749979


### Singular values depend on both inner products

For Gram matrices `M` (source) and `N` (target), factor `M = Lm Lm^T` and
`N = Ln Ln^T`. The singular values of `A` are those of the array
`Ln^T A Lm^{-T}`, which is the matrix of `A` in orthonormal bases. They are
also the square roots of the eigenvalues of `A* A = M^{-1} A^T N A`.

In [8]:
def metric_svd(A, M, N):
    Lm = torch.linalg.cholesky(M)
    Ln = torch.linalg.cholesky(N)
    core = Ln.T @ A @ torch.linalg.inv(Lm).T
    Uc, s, Vhc = torch.linalg.svd(core, full_matrices=False)
    W = torch.linalg.inv(Ln).T @ Uc        # columns w_i, N-orthonormal
    V = torch.linalg.inv(Lm).T @ Vhc.T     # columns v_i, M-orthonormal
    return W, s, V

M = torch.diag(torch.tensor([1.0, 4.0]))
N = torch.tensor([[2.0, 1.0], [1.0, 3.0]])
W, s_MN, V = metric_svd(A, M, N)
Astar = torch.linalg.solve(M, A.T @ N)
eig = torch.linalg.eigvals(Astar @ A).real.sort(descending=True).values
print("singular values for (M, N):", [round(v, 5) for v in s_MN.tolist()])
print("sqrt eig(A* A)            :", [round(v, 5) for v in eig.sqrt().tolist()])
assert torch.allclose(s_MN, eig.sqrt())
assert torch.allclose(V.T @ M @ V, torch.eye(2), atol=1e-12)
assert torch.allclose(W.T @ N @ W, torch.eye(2), atol=1e-12)
assert torch.allclose(A @ V, W * s_MN)          # A v_i = sigma_i w_i
assert torch.allclose(Astar @ W, V * s_MN)      # A* w_i = sigma_i v_i
print("not the singular values for the dot products:", s.tolist())

singular values for (M, N): [10.30045, 1.62813]
sqrt eig(A* A)            : [10.30045, 1.62813]
not the singular values for the dot products: [6.70820393249937, 2.2360679774997894]


## 6. Pseudo-inverse and least squares by gradient descent

Exercise 5.6: `A = [1 1]`, for the dot product and for `M = diag(1, 4)`.
Proposition 5.18: the iteration `x - eta A*(A x - b)` converges exactly when
`eta < 2 / sigma_1^2`, to the solution closest to the start.

In [9]:
A = torch.tensor([[1.0, 1.0]])
M = torch.diag(torch.tensor([1.0, 4.0]))

def pinv_metric(A, M):
    Astar = torch.linalg.solve(M, A.T)          # N = identity on R
    return Astar @ torch.linalg.inv(A @ Astar)

p_dot = pinv_metric(A, torch.eye(2))
p_M = pinv_metric(A, M)
print("A+ for the dot product :", p_dot.flatten().tolist())
print("A+ for M = diag(1, 4)  :", p_M.flatten().tolist())
assert torch.allclose(p_dot, torch.linalg.pinv(A))
assert torch.allclose(p_M.flatten(), torch.tensor([0.8, 0.2]))

# closest solution to x0 = (0.5, -0.5) for y = 2 (Exercise 3.8)
x0 = torch.tensor([0.5, -0.5]); y = torch.tensor([2.0])
for name, P in (("dot", p_dot), ("M", p_M)):
    print(f"closest solution to x0 ({name}):",
          [round(v, 6) for v in (x0 + P @ (y - A @ x0)).tolist()])
assert torch.allclose(x0 + p_dot @ (y - A @ x0), torch.tensor([1.5, 0.5]))
assert torch.allclose(x0 + p_M @ (y - A @ x0), torch.tensor([2.1, -0.1]))

A+ for the dot product : [0.5, 0.5]
A+ for M = diag(1, 4)  : [0.8, 0.2]
closest solution to x0 (dot): [1.5, 0.5]
closest solution to x0 (M): [2.1, -0.1]


In [10]:
# an underdetermined problem: 3 equations, 6 unknowns, rank 3
torch.manual_seed(1)
A = torch.randn(3, 6)
b = torch.randn(3)
x0 = torch.randn(6)
sig = torch.linalg.svdvals(A)
eta_max = 2 / sig[0].item() ** 2
x_inf = x0 + torch.linalg.pinv(A) @ (b - A @ x0)
print("sigma:", [round(v, 4) for v in sig.tolist()],
      " 2/sigma_1^2 =", round(eta_max, 5))

def gd(eta, steps):
    x = x0.clone()
    for _ in range(steps):
        x = x - eta * A.T @ (A @ x - b)
    return x

for factor in (0.5, 0.99, 1.01):
    x = gd(factor * eta_max, 4000)
    print(f"eta = {factor:4.2f} * 2/sigma_1^2 : distance to x_inf =",
          f"{(x - x_inf).norm().item():.3e}")
assert (gd(0.5 * eta_max, 4000) - x_inf).norm() < 1e-10
assert (gd(0.99 * eta_max, 4000) - x_inf).norm() < 1e-6
assert (gd(1.01 * eta_max, 4000) - x_inf).norm() > 1e3

# rate: max(|1 - eta s1^2|, |1 - eta sr^2|)
eta = 0.5 * eta_max
rate = max(abs(1 - eta * sig[0].item() ** 2), abs(1 - eta * sig[-1].item() ** 2))
e = [(gd(eta, k) - x_inf).norm().item() for k in (10, 11)]
print("observed ratio:", round(e[1] / e[0], 6), " predicted bound:", round(rate, 6))
assert e[1] / e[0] <= rate + 1e-9

sigma: [4.4182, 3.5654, 2.5318]  2/sigma_1^2 = 0.10246


eta = 0.50 * 2/sigma_1^2 : distance to x_inf = 5.596e-16


eta = 0.99 * 2/sigma_1^2 : distance to x_inf = 1.692e-15
eta = 1.01 * 2/sigma_1^2 : distance to x_inf = 2.384e+34


observed ratio: 0.671612  predicted bound: 0.671613


## 7. Eckart–Young–Mirsky and Weyl's inequalities

Theorem 5.20: no matrix of rank `k` is closer to `A` than the truncated
SVD, in the operator norm and in the Hilbert–Schmidt (Frobenius) norm.
Proposition 5.15: singular values move by at most the norm of a
perturbation.

In [11]:
torch.manual_seed(2)
A = torch.randn(8, 6)
U, s, Vh = torch.linalg.svd(A, full_matrices=False)
k = 2
Ak = (U[:, :k] * s[:k]) @ Vh[:k]
err_op = torch.linalg.matrix_norm(A - Ak, ord=2).item()
err_hs = torch.linalg.matrix_norm(A - Ak).item()
print("||A - A_k||    =", round(err_op, 6), " sigma_{k+1} =", round(s[k].item(), 6))
print("||A - A_k||_HS =", round(err_hs, 6),
      " sqrt(sum of dropped squares) =", round(s[k:].square().sum().sqrt().item(), 6))
assert abs(err_op - s[k].item()) < 1e-12
assert abs(err_hs - s[k:].square().sum().sqrt().item()) < 1e-12

# random rank-k competitors, and competitors close to A_k
worst_op, worst_hs = float("inf"), float("inf")
for _ in range(3000):
    B = torch.randn(8, k) @ torch.randn(k, 6)
    C = Ak + 0.05 * torch.randn(8, k) @ torch.randn(k, 6)
    Uc, sc, Vhc = torch.linalg.svd(C, full_matrices=False)
    C = (Uc[:, :k] * sc[:k]) @ Vhc[:k]            # back to rank k
    for X in (B, C):
        worst_op = min(worst_op, torch.linalg.matrix_norm(A - X, ord=2).item())
        worst_hs = min(worst_hs, torch.linalg.matrix_norm(A - X).item())
print("best competitor: operator", round(worst_op, 6), " HS", round(worst_hs, 6))
assert worst_op >= err_op - 1e-12 and worst_hs >= err_hs - 1e-12

# Proposition 5.15: |sigma_k(A) - sigma_k(B)| <= ||A - B||, and
# sigma_{i+j}(A) <= sigma_i(A - B) when rank B <= j
for _ in range(500):
    E = 0.3 * torch.randn(8, 6)
    d = (torch.linalg.svdvals(A + E) - s).abs().max().item()
    assert d <= torch.linalg.matrix_norm(E, ord=2).item() + 1e-12
    j = 2
    B = torch.randn(8, j) @ torch.randn(j, 6)
    sAB = torch.linalg.svdvals(A - B)
    assert torch.all(s[j:] <= sAB[: 6 - j] + 1e-12)
print("perturbation and interlacing inequalities hold on 500 random draws")

# Exercise 5.9: singular values 5, 3, 1
D = torch.diag(torch.tensor([5.0, 3.0, 1.0]))
for kk in (1, 2):
    Dk = D.clone(); Dk[kk:, kk:] = 0
    e_op = torch.linalg.matrix_norm(D - Dk, ord=2).item()
    e_hs = torch.linalg.matrix_norm(D - Dk).item()
    print(f"Exercise 5.9: rank <= {kk}: errors", round(e_op, 4), round(e_hs, 4))
    assert abs(e_op - (3.0, 1.0)[kk - 1]) < 1e-12
    assert abs(e_hs - (math.sqrt(10), 1.0)[kk - 1]) < 1e-12

# Exercise 5.7: Hilbert-Schmidt norm between ||A|| and sqrt(r) ||A||, LoRA count
hs, op = torch.linalg.matrix_norm(A).item(), s[0].item()
assert op <= hs <= math.sqrt(6) * op
print("LoRA rank 8 on 1000 x 1000:", 8 * (1000 + 1000), "numbers,",
      f"{8 * 2000 / 1e6:.1%} of the full matrix")

||A - A_k||    = 3.206335  sigma_{k+1} = 3.206335
||A - A_k||_HS = 3.660726  sqrt(sum of dropped squares) = 3.660726


best competitor: operator 3.206343  HS 3.661056
perturbation and interlacing inequalities hold on 500 random draws
Exercise 5.9: rank <= 1: errors 3.0 3.1623
Exercise 5.9: rank <= 2: errors 1.0 1.0
LoRA rank 8 on 1000 x 1000: 16000 numbers, 1.6% of the full matrix


## 8. PCA depends on the inner product

Exercise 5.8 and the machine-learning thread of Section 5.4. We draw data
with covariance `[[1, 0.5], [0.5, 1]]`, then re-express the first feature in
a unit ten times smaller. The principal direction changes; it is the PCA of
the original data for the inner product `M = diag(100, 1)`.

In [12]:
C_hat = torch.tensor([[1.0, 0.5], [0.5, 1.0]])
lam, E = torch.linalg.eigh(C_hat)
print("dot product: eigenvalues", lam.flip(0).tolist(),
      " direction", (E[:, 1] * torch.sign(E[0, 1])).tolist(),
      " fraction", (lam[1] / lam.sum()).item())
assert torch.allclose(lam, torch.tensor([0.5, 1.5]))

T = torch.diag(torch.tensor([10.0, 1.0]))        # new coordinates y = T x
C_new = T @ C_hat @ T.T
lam2, E2 = torch.linalg.eigh(C_new)
d_new = E2[:, 1] * torch.sign(E2[0, 1])
print("rescaled   : array", C_new.tolist())
print("             eigenvalues", [round(v, 3) for v in lam2.flip(0).tolist()],
      " direction", [round(v, 4) for v in d_new.tolist()],
      " fraction", round((lam2[1] / lam2.sum()).item(), 4))
assert torch.allclose(C_new, torch.tensor([[100.0, 5.0], [5.0, 1.0]]))
assert torch.allclose(lam2, torch.tensor([(101 - math.sqrt(9901)) / 2,
                                          (101 + math.sqrt(9901)) / 2]))

# the covariance operator for M = diag(100, 1) in the original coordinates
M = T.T @ T
op = C_hat @ M
lam3, V3 = torch.linalg.eig(op)
i = lam3.real.argmax()
d_orig = V3[:, i].real
d_orig = d_orig / d_orig[0] * 0.5
print("C_hat M =", op.tolist(), " eigenvalues",
      [round(v, 3) for v in lam3.real.sort(descending=True).values.tolist()])
print("first eigenvector, original coordinates:", [round(v, 4) for v in d_orig.tolist()])
assert torch.allclose(lam3.real.sort().values, lam2)
assert torch.allclose(T @ d_orig / (T @ d_orig).norm(), d_new)

dot product: eigenvalues [1.5, 0.5]  direction [0.7071067811865475, 0.7071067811865475]  fraction 0.75
rescaled   : array [[100.0, 5.0], [5.0, 1.0]]
             eigenvalues [100.252, 0.748]  direction [0.9987, 0.0503]  fraction 0.9926
C_hat M = [[100.0, 0.5], [50.0, 1.0]]  eigenvalues [100.252, 0.748]
first eigenvector, original coordinates: [0.5, 0.2519]


In [13]:
# the same with sampled data and the statement of Proposition 5.21
torch.manual_seed(3)
Nn = 4000
X = torch.randn(Nn, 2) @ torch.linalg.cholesky(C_hat).T
X = X - X.mean(0)
D = X.T / math.sqrt(Nn)                    # columns u_j / sqrt(N)
U, s, _ = torch.linalg.svd(D, full_matrices=False)
C_emp = X.T @ X / Nn
print("empirical covariance:", C_emp.round(decimals=3).tolist())
assert torch.allclose(D @ D.T, C_emp)
assert torch.allclose(s.square(), torch.linalg.eigvalsh(C_emp).flip(0))

def mean_sq_dist(direction):
    u = direction / direction.norm()
    R = X - torch.outer(X @ u, u)
    return R.square().sum(1).mean().item()

best = mean_sq_dist(U[:, 0])
angles = torch.linspace(0, math.pi, 721)
scan = min(mean_sq_dist(torch.tensor([math.cos(a), math.sin(a)])) for a in angles)
print("mean squared distance to the first principal line:", round(best, 6))
print("smallest eigenvalue of the covariance            :", round(s[1].item() ** 2, 6))
print("best line found by scanning all angles           :", round(scan, 6))
assert abs(best - s[1].item() ** 2) < 1e-12 and scan >= best - 1e-12

empirical covariance: [[0.985, 0.538], [0.538, 1.061]]
mean squared distance to the first principal line: 0.483056
smallest eigenvalue of the covariance            : 0.483056
best line found by scanning all angles           : 0.483056


## Summary

* A quadratic form keeps its signature under `P^T G P`; its eigenvalues are
  those of `M^{-1} G` and depend on the inner product `M`.
* Gradient descent on a quadratic converges for `eta < 2 / lambda_1` and at
  best at the rate `(kappa - 1) / (kappa + 1)`; the inner product `M = H`
  gives `kappa = 1`.
* Singular values depend on both inner products; the pseudo-inverse and the
  limit of gradient descent on least squares depend on the inner product of
  parameter space and on the starting point.
* Truncating the SVD is the best low-rank approximation, and PCA is that
  statement for the data map, for a chosen inner product.